# DuST — UCI HAR, CNN baseline + Swin Transformer 1D

Notebook độc lập, không cần clone repo/private token. Bật GPU nếu tài khoản còn quota;
CPU cũng chạy được. Input chỉ `total_acc_x/y/z`, 50 Hz, 128 điểm, 6 lớp.
Train/validation/test không trùng subject; normalization fit train-only.
Checkpoint chọn bằng **validation macro-F1**. Test chỉ mở ở bước cuối sau khi đóng băng lựa chọn.

Nguồn: [UCI HAR chính thức](https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones).
Bật Internet để tải ZIP, hoặc Add Input ZIP/folder gốc rồi sửa `DATA_SOURCE`.
Không tải checkpoint không rõ nguồn: `torch.load` dùng pickle cho artifact tự tạo.
Mặc định một seed để tiết kiệm quota; muốn ba seed phải chọn trước khi chạy evaluation.
Notebook chưa có kết quả GPU Kaggle nhúng sẵn; tất cả output sinh từ lần chạy của bạn.


In [ ]:
import os, sys, json, subprocess, zipfile
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
# Chỉ sửa cấu hình trước khi chạy. Không thay candidates sau khi xem test.
WORK_ROOT = Path(os.environ.get('DUST_WORK_ROOT', '/kaggle/working/dust-project'))
DATA_SOURCE = os.environ.get('DUST_DATA_SOURCE') or None  # ZIP hoặc folder UCI HAR gốc
SEEDS = [42]  # Có quota: [42, 43, 44], chốt trước evaluation.
DEVICE = 'auto'  # Kaggle GPU nếu khả dụng, nếu không CPU.
if not SEEDS or len(set(SEEDS)) != len(SEEDS):
    raise ValueError('SEEDS must be non-empty and unique')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=WORK_ROOT, check=True)


## 1. Khôi phục mã nguồn/config đã đóng gói
Không ghi đè một file có nội dung khác: dùng thư mục mới nếu đã chỉnh source ở run trước.


In [ ]:
SOURCES = {'models/SwinTransformer_cls.py': '# -*- coding: utf-8 -*-\n"""\n@File    :  SWT_V2_1D.py\n@Time    :  2022/8/14 16:15\n@Author  :  HUANG Dong\n@Version :  0.0.1\n@Contact :  huang_dong@tju.edu.cn\n@Desc    :  None\n"""\nimport math\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.utils.checkpoint as checkpoint\nfrom torch.nn.init import trunc_normal_\nimport numpy as np\n\n\ndef drop_path(x, drop_prob: float = 0., training: bool = False, scale_by_keep: bool = True):\n    """Drop paths (Stochastic Depth) per sample (when applied in main path of residual blocks).\n\n    This is the same as the DropConnect impl I created for EfficientNet, etc networks, however,\n    the original name is misleading as \'Drop Connect\' is a different form of dropout in a separate paper...\n    See discussion: https://github.com/tensorflow/tpu/issues/494#issuecomment-532968956 ... I\'ve opted for\n    changing the layer and argument names to \'drop path\' rather than mix DropConnect as a layer name and use\n    \'survival rate\' as the argument.\n\n    """\n    if drop_prob == 0. or not training:\n        return x\n    keep_prob = 1 - drop_prob\n    shape = (x.shape[0],) + (1,) * (x.ndim - 1)  # work with diff dim tensors, not just 2D ConvNets\n    random_tensor = x.new_empty(shape).bernoulli_(keep_prob)\n    if keep_prob > 0.0 and scale_by_keep:\n        random_tensor.div_(keep_prob)\n    return x * random_tensor\n\n\nclass DropPath(nn.Module):\n    """Drop paths (Stochastic Depth) per sample  (when applied in main path of residual blocks).\n    """\n    def __init__(self, drop_prob: float = 0., scale_by_keep: bool = True):\n        super(DropPath, self).__init__()\n        self.drop_prob = drop_prob\n        self.scale_by_keep = scale_by_keep\n\n    def forward(self, x):\n        return drop_path(x, self.drop_prob, self.training, self.scale_by_keep)\n\n    def extra_repr(self):\n        return f\'drop_prob={round(self.drop_prob,3):0.3f}\'\n\n\nclass Mlp(nn.Module):\n    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):\n        super().__init__()\n        out_features = out_features or in_features\n        hidden_features = hidden_features or in_features\n        self.fc1 = nn.Linear(in_features, hidden_features)\n        self.act = act_layer()\n        self.fc2 = nn.Linear(hidden_features, out_features)\n        self.drop = nn.Dropout(drop)\n\n    def forward(self, x):\n        x = self.fc1(x)\n        x = self.act(x)\n        x = self.drop(x)\n        x = self.fc2(x)\n        x = self.drop(x)\n        return x\n\n\ndef window_partition(x, window_size):\n    """\n    Args:\n        x: (B, L, C)\n        window_size (int): window size\n\n    Returns:\n        windows: (num_windows*B, window_size, C)\n    """\n    B, L, C = x.shape\n    x = x.view(B, L // window_size, window_size, C)\n    windows = x.permute(0, 1, 2, 3).contiguous().view(-1, window_size, C)\n    return windows\n\n\ndef window_reverse(windows, window_size, L):\n    """\n    Args:\n        windows: (num_windows*B, window_size, window_size, C)\n        window_size (int): Window size\n        L (int): sequence length\n\n    Returns:\n        x: (B, L, C)\n    """\n    B = int(windows.shape[0] / (L / window_size))\n    x = windows.view(B, L // window_size, window_size, -1)\n    x = x.permute(0, 1, 2, 3).contiguous().view(B, L, -1)\n    return x\n\n\n# class WindowAttention_1D(nn.Module):\n#     r""" Window based multi-head self attention (W-MSA) module with relative position bias.\n#     It supports both of shifted and non-shifted window.\n#     Args:\n#         dim (int): Number of input channels.\n#         window_size (int): The height and width of the window.\n#         num_heads (int): Number of attention heads.\n#         qkv_bias (bool, optional):  If True, add a learnable bias to query, key, value. Default: True\n#         attn_drop (float, optional): Dropout ratio of attention weight. Default: 0.0\n#         proj_drop (float, optional): Dropout ratio of output. Default: 0.0\n#         pretrained_window_size (int): The height and width of the window in pre-training.\n#     """\n\n#     def __init__(self, dim, window_size, num_heads, qkv_bias=True, attn_drop=0., proj_drop=0.,\n#                  pretrained_window_size=0):\n\n#         super().__init__()\n#         self.dim = dim\n#         self.window_size = window_size  # Wl\n#         self.pretrained_window_size = pretrained_window_size\n#         self.num_heads = num_heads\n\n#         self.logit_scale = nn.Parameter(torch.log(10 * torch.ones((num_heads, 1, 1))), requires_grad=True)\n\n#         # mlp to generate continuous relative position bias\n#         self.cpb_mlp = nn.Sequential(nn.Linear(1, 512, bias=True),\n#                                      nn.ReLU(inplace=True),\n#                                      nn.Linear(512, num_heads, bias=False))\n\n#         # get relative_coords_table\n#         relative_coords_l = torch.arange(-(self.window_size - 1), self.window_size, dtype=torch.float32)\n#         relative_coords_table = torch.stack(\n#             torch.meshgrid([relative_coords_l], indexing=\'ij\')).permute(1, 0).contiguous().unsqueeze(0)  # 1, 2*Wl-1, 1\n#         if pretrained_window_size > 0:\n#             relative_coords_table[:, :, :] /= (pretrained_window_size - 1)\n#         else:\n#             relative_coords_table[:, :, :] /= (self.window_size - 1)\n#         relative_coords_table *= 8  # normalize to -8, 8\n#         relative_coords_table = torch.sign(relative_coords_table) * torch.log2(\n#             torch.abs(relative_coords_table) + 1.0) / np.log2(8)\n\n#         self.register_buffer("relative_coords_table", relative_coords_table)\n\n#         # get pair-wise relative position index for each token inside the window\n#         coords_l = torch.arange(self.window_size)\n#         coords = torch.stack(torch.meshgrid([coords_l], indexing=\'ij\'))  # 1, Wl\n#         coords_flatten = torch.flatten(coords, 1)  # 1, Wl\n#         relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]  # 1, Wl, Wl\n#         relative_coords = relative_coords.permute(1, 2, 0).contiguous()  # Wl, Wl, 1\n#         relative_coords[:, :, 0] += self.window_size - 1  # shift to start from 0\n#         relative_position_index = relative_coords.sum(-1)  # Wl, Wl\n#         self.register_buffer("relative_position_index", relative_position_index)\n\n#         self.qkv = nn.Linear(dim, dim * 3, bias=False)\n#         if qkv_bias:\n#             self.q_bias = nn.Parameter(torch.zeros(dim))\n#             self.v_bias = nn.Parameter(torch.zeros(dim))\n#         else:\n#             self.q_bias = None\n#             self.v_bias = None\n#         self.attn_drop = nn.Dropout(attn_drop)\n#         self.proj = nn.Linear(dim, dim)\n#         self.proj_drop = nn.Dropout(proj_drop)\n#         self.softmax = nn.Softmax(dim=-1)\n\n#     def forward(self, x, mask=None):\n#         """\n#         Args:\n#             x: input features with shape of (num_windows*B, N, C)\n#             mask: (0/-inf) mask with shape of (num_windows, Wl, Wl) or None\n#         """\n#         B_, N, C = x.shape\n#         qkv_bias = None\n#         if self.q_bias is not None:\n#             qkv_bias = torch.cat((self.q_bias, torch.zeros_like(self.v_bias, requires_grad=False), self.v_bias))\n#         qkv = F.linear(input=x, weight=self.qkv.weight, bias=qkv_bias)\n#         qkv = qkv.reshape(B_, N, 3, self.num_heads, -1).permute(2, 0, 3, 1, 4)\n#         q, k, v = qkv[0], qkv[1], qkv[2]  # make torchscript happy (cannot use tensor as tuple)\n\n#         # cosine attention\n#         attn = (F.normalize(q, dim=-1) @ F.normalize(k, dim=-1).transpose(-2, -1))\n#         logit_scale = torch.clamp(self.logit_scale, max=torch.log(torch.tensor(1. / 0.01, device=attn.device))).exp()\n#         attn = attn * logit_scale\n\n#         relative_position_bias_table = self.cpb_mlp(self.relative_coords_table).view(-1, self.num_heads)\n#         relative_position_bias = relative_position_bias_table[self.relative_position_index.view(-1)].view(\n#             self.window_size, self.window_size, -1)  # Wl,l,nH\n#         relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()  # nH, Wl, Wl\n#         relative_position_bias = 16 * torch.sigmoid(relative_position_bias)\n#         attn = attn + relative_position_bias.unsqueeze(0)\n\n#         if mask is not None:\n#             nW = mask.shape[0]\n#             attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)\n#             attn = attn.view(-1, self.num_heads, N, N)\n#             attn = self.softmax(attn)\n#         else:\n#             attn = self.softmax(attn)\n\n#         attn = self.attn_drop(attn)\n\n#         x = (attn @ v).transpose(1, 2).reshape(B_, N, C)\n#         x = self.proj(x)\n#         x = self.proj_drop(x)\n#         return x\n\n###\nclass WindowAttention1D(nn.Module):\n    r""" Window based multi-head self attention (W-MSA) module with relative position bias.\n    It supports both of shifted and non-shifted window.\n\n    Args:\n        dim (int): Number of input channels.\n        window_size (int): The height and width of the window.\n        num_heads (int): Number of attention heads.\n        qkv_bias (bool, optional):  If True, add a learnable bias to query, key, value. Default: True\n        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set\n        attn_drop (float, optional): Dropout ratio of attention weight. Default: 0.0\n        proj_drop (float, optional): Dropout ratio of output. Default: 0.0\n    """\n\n    def __init__(self, dim, window_size, num_heads, qkv_bias=True, qk_scale=None, attn_drop=0., proj_drop=0.):\n\n        super().__init__()\n        self.dim = dim\n        self.window_size = window_size  # Wl\n        self.num_heads = num_heads\n        head_dim = dim // num_heads\n        self.scale = qk_scale or head_dim ** -0.5\n\n        # define a parameter table of relative position bias\n        self.relative_position_bias_table = nn.Parameter(\n            torch.zeros((2 * window_size - 1), num_heads))  # 2*Ww-1, nH\n\n        # get pair-wise relative position index for each token inside the window\n        coords_l = torch.arange(self.window_size)\n        coords = torch.stack(torch.meshgrid(\n            [coords_l], indexing=\'ij\'))  # 1, Wl\n        coords_flatten = torch.flatten(coords, 1)  # 1, Wl\n        relative_coords = coords_flatten[:, :, None] - \\\n            coords_flatten[:, None, :]  # 1, Wl, Wl\n        relative_coords = relative_coords.permute(\n            1, 2, 0).contiguous()  # Wl, Wl, 2\n        relative_coords[:, :, 0] += self.window_size - \\\n            1  # shift to start from 0\n        # relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1\n        relative_position_index = relative_coords.sum(-1)  # Wl, Wl\n        self.register_buffer("relative_position_index",\n                             relative_position_index)\n\n        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)\n        self.attn_drop = nn.Dropout(attn_drop)\n        self.proj = nn.Linear(dim, dim)\n        self.proj_drop = nn.Dropout(proj_drop)\n\n        trunc_normal_(self.relative_position_bias_table, std=.02)\n        self.softmax = nn.Softmax(dim=-1)\n\n    def forward(self, x, mask=None):\n        """\n        Args:\n            x: input features with shape of (num_windows*B, N, C)\n            mask: (0/-inf) mask with shape of (num_windows, Wl, Wl) or None\n        """\n        B_, N, C = x.shape\n        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C //\n                                  self.num_heads).permute(2, 0, 3, 1, 4)\n        # make torchscript happy (cannot use tensor as tuple)\n        q, k, v = qkv[0], qkv[1], qkv[2]\n\n        q = q * self.scale\n        attn = (q @ k.transpose(-2, -1))\n\n        relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(\n            self.window_size, self.window_size, -1)  # Wl,Wl,nH\n        relative_position_bias = relative_position_bias.permute(\n            2, 0, 1).contiguous()  # nH, Wl, Wl\n        attn = attn + relative_position_bias.unsqueeze(0)\n\n        if mask is not None:\n            nW = mask.shape[0]\n            attn = attn.view(B_ // nW, nW, self.num_heads, N,\n                             N) + mask.unsqueeze(1).unsqueeze(0)\n            attn = attn.view(-1, self.num_heads, N, N)\n            attn = self.softmax(attn)\n        else:\n            attn = self.softmax(attn)\n\n        attn = self.attn_drop(attn)\n\n        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)\n        x = self.proj(x)\n        x = self.proj_drop(x)\n        return x\n###\n\ndef compute_mask(L, window_size, shift_size):\n    Lp = int(np.ceil(L / window_size)) * window_size\n    img_mask = torch.zeros((1, Lp, 1))  # 1 Lp 1\n    pad_size = int(Lp - L)\n    if (pad_size == 0) or (pad_size + shift_size == window_size):\n        segs = (slice(-window_size), slice(-window_size, -shift_size), slice(-shift_size, None))\n    elif pad_size + shift_size > window_size:\n        seg1 = int(window_size * 2 - L + shift_size)\n        segs = (\n            slice(-seg1), slice(-seg1, -window_size), slice(-window_size, -shift_size), slice(-shift_size, None))\n    elif pad_size + shift_size < window_size:\n        seg1 = int(window_size * 2 - L + shift_size)\n        segs = (\n            slice(-window_size), slice(-window_size, -seg1), slice(-seg1, -shift_size), slice(-shift_size, None))\n    cnt = 0\n    for d in segs:\n        img_mask[:, d, :] = cnt\n        cnt += 1\n    mask_windows = window_partition(img_mask, window_size)  # nW, ws, 1\n    mask_windows = mask_windows.squeeze(-1)  # nW, ws\n    attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)\n    attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))\n    return attn_mask\n\n\nclass SwinTransformerBlock_1D(nn.Module):\n    r""" Swin Transformer Block.\n    Args:\n        dim (int): Number of input channels.\n        input_resolution (int): Input resulotion.\n        num_heads (int): Number of attention heads.\n        window_size (int): Window size.\n        shift_size (int): Shift size for SW-MSA.\n        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim.\n        qkv_bias (bool, optional): If True, add a learnable bias to query, key, value. Default: True\n        drop (float, optional): Dropout rate. Default: 0.0\n        attn_drop (float, optional): Attention dropout rate. Default: 0.0\n        drop_path (float, optional): Stochastic depth rate. Default: 0.0\n        act_layer (nn.Module, optional): Activation layer. Default: nn.GELU\n        norm_layer (nn.Module, optional): Normalization layer.  Default: nn.LayerNorm\n        pretrained_window_size (int): Window size in pre-training.\n    """\n\n    def __init__(self, dim, input_resolution, num_heads, window_size=7, shift_size=0,\n                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0., drop_path=0.,\n                 act_layer=nn.GELU, norm_layer=nn.LayerNorm, pretrained_window_size=0):\n        super().__init__()\n        self.dim = dim\n        self.input_resolution = input_resolution\n        self.num_heads = num_heads\n        self.window_size = window_size\n        self.shift_size = shift_size\n        self.mlp_ratio = mlp_ratio\n        if self.input_resolution <= self.window_size:\n            # if window size is larger than input resolution, we don\'t partition windows\n            self.shift_size = 0\n            self.window_size = self.input_resolution\n        assert 0 <= self.shift_size < self.window_size, "shift_size must in 0-window_size"\n\n        self.norm1 = norm_layer(dim)\n        # self.attn = WindowAttention_1D(\n        #     dim, window_size=self.window_size, num_heads=num_heads,\n        #     qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop,\n        #     pretrained_window_size=pretrained_window_size)\n        self.attn = WindowAttention1D(\n            dim, window_size=self.window_size, num_heads=num_heads,\n            qkv_bias=qkv_bias, qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=drop)\n\n        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()\n        self.norm2 = norm_layer(dim)\n        mlp_hidden_dim = int(dim * mlp_ratio)\n        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)\n\n        if self.shift_size > 0:\n            # calculate attention mask for SW-MSA\n            L = self.input_resolution\n            attn_mask = compute_mask(L, self.window_size, self.shift_size)\n        else:\n            attn_mask = None\n\n        self.register_buffer("attn_mask", attn_mask)\n\n    def forward(self, x):\n        Li = self.input_resolution\n        B, L, C = x.shape\n        assert L == Li, "input feature has wrong size"\n\n        shortcut = x\n        # x = x.view(B, L, C)\n\n        # padding x\n        pad_r = (self.window_size - L % self.window_size) % self.window_size\n        x = F.pad(x, (0, 0, 0, pad_r))\n        _, Lp, _ = x.shape\n\n        # cyclic shift\n        if self.shift_size > 0:\n            shifted_x = torch.roll(x, shifts=(-self.shift_size), dims=(1))\n        else:\n            shifted_x = x\n\n        # partition windows\n        x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, C\n        x_windows = x_windows.view(-1, self.window_size, C)  # nW*B, window_siz, C\n\n        # W-MSA/SW-MSA\n        attn_windows = self.attn(x_windows, mask=self.attn_mask)  # nW*B, window_size, C\n\n        # merge windows\n        attn_windows = attn_windows.view(-1, self.window_size, C)\n        shifted_x = window_reverse(attn_windows, self.window_size, Lp)  # B L\' C\n\n        # reverse cyclic shift\n        if self.shift_size > 0:\n            x = torch.roll(shifted_x, shifts=(self.shift_size), dims=(1))\n        else:\n            x = shifted_x\n        x = x.view(B, Lp, C)\n        # reverse padding x\n        x = x[:, :L, :].contiguous()\n        x = shortcut + self.drop_path(self.norm1(x))\n\n        # FFN\n        x = x + self.drop_path(self.norm2(self.mlp(x)))\n\n        return x\n\n\nclass PatchMerging(nn.Module):\n    """ Patch Merging Layer\n\n    Args:\n        dim (int): Number of input channels.\n        norm_layer (nn.Module, optional): Normalization layer.  Default: nn.LayerNorm\n    """\n    def __init__(self, dim, norm_layer=nn.LayerNorm):\n        super().__init__()\n        self.dim = dim\n        self.reduction = nn.Linear(2 * dim, 2 * dim, bias=False)\n        self.norm = norm_layer(2 * dim)\n\n    def forward(self, x):\n        """ Forward function.\n\n        Args:\n            x: Input feature, tensor size (B, L, C).\n        """\n        B, L, C = x.shape\n\n        # padding\n        # pad_input = (L % 2 == 1)\n        # if pad_input:\n        #     x = F.pad(x, (0, 0, 0, L % 2))\n        x = F.pad(x, (0, 0, 0, L % 2))\n\n        x0 = x[:, 0::2, :]  # B L/2 C\n        x1 = x[:, 1::2, :]  # B L/2 C\n        x = torch.cat([x0, x1], -1)  # B L/2 2*C\n\n        x = self.norm(x)\n        x = self.reduction(x)\n\n        return x\n\n\nclass BasicLayer(nn.Module):\n    """ A basic Swin Transformer layer for one stage.\n    Args:\n        dim (int): Number of input channels.\n        input_resolution (int): Input resolution.\n        depth (int): Number of blocks.\n        num_heads (int): Number of attention heads.\n        window_size (int): Local window size.\n        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim.\n        qkv_bias (bool, optional): If True, add a learnable bias to query, key, value. Default: True\n        drop (float, optional): Dropout rate. Default: 0.0\n        attn_drop (float, optional): Attention dropout rate. Default: 0.0\n        drop_path (float | tuple[float], optional): Stochastic depth rate. Default: 0.0\n        norm_layer (nn.Module, optional): Normalization layer. Default: nn.LayerNorm\n        downsample (nn.Module | None, optional): Downsample layer at the end of the layer. Default: None\n        use_checkpoint (bool): Whether to use checkpointing to save memory. Default: False.\n        pretrained_window_size (int): Local window size in pre-training.\n    """\n\n    def __init__(self, dim, input_resolution, depth, num_heads, window_size,\n                 mlp_ratio=4., qkv_bias=True, drop=0., attn_drop=0.,\n                 drop_path=0., norm_layer=nn.LayerNorm, downsample=None, use_checkpoint=False,\n                 pretrained_window_size=0):\n\n        super().__init__()\n        self.dim = dim\n        self.input_resolution = input_resolution\n        self.depth = depth\n        self.use_checkpoint = use_checkpoint\n\n        # build blocks\n        self.blocks = nn.ModuleList([\n            SwinTransformerBlock_1D(dim=dim, input_resolution=input_resolution,\n                                 num_heads=num_heads, window_size=window_size,\n                                 shift_size=0 if (i % 2 == 0) else window_size // 2,\n                                 mlp_ratio=mlp_ratio,\n                                 qkv_bias=qkv_bias,\n                                 drop=drop, attn_drop=attn_drop,\n                                 drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,\n                                 norm_layer=norm_layer,\n                                 pretrained_window_size=pretrained_window_size)\n            for i in range(depth)])\n\n        # patch merging layer\n        if downsample is not None:\n            self.downsample = downsample(dim=dim, norm_layer=norm_layer)\n        else:\n            self.downsample = None\n\n    def forward(self, x):\n        for blk in self.blocks:\n            if self.use_checkpoint:\n                x = checkpoint.checkpoint(blk, x)\n            else:\n                x = blk(x)\n        if self.downsample is not None:\n            x = self.downsample(x)\n        return x\n\n    def _init_respostnorm(self):\n        for blk in self.blocks:\n            nn.init.constant_(blk.norm1.bias, 0)\n            nn.init.constant_(blk.norm1.weight, 0)\n            nn.init.constant_(blk.norm2.bias, 0)\n            nn.init.constant_(blk.norm2.weight, 0)\n\n\nclass PatchEmbed1D(nn.Module):\n    """ Video to Patch Embedding.\n\n    Args:\n        seq_len (int): Sequence length.\n        patch_size (int): Patch token size. Default: 4.\n        in_chans (int): Number of input video channels. Default: 3.\n        embed_dim (int): Number of linear projection output channels. Default: 96.\n        norm_layer (nn.Module, optional): Normalization layer. Default: None\n    """\n    def __init__(self, seq_len, patch_size=4, in_chans=32, embed_dim=128, norm_layer=None):\n        super().__init__()\n        self.patch_size = patch_size\n\n        self.in_chans = in_chans\n        self.embed_dim = embed_dim\n\n        self.proj = nn.Conv1d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)\n        self.num_patches = math.ceil(seq_len / patch_size)\n        self.patches_resolution = math.ceil(seq_len / patch_size)\n        if norm_layer is not None:\n            self.norm = norm_layer(embed_dim)\n        else:\n            self.norm = None\n\n    def forward(self, x):\n        """Forward function."""\n        # padding\n        _, _, L = x.size()\n        pad_r = (self.patch_size - L % self.patch_size) % self.patch_size\n        x = F.pad(x, (0, pad_r))\n        x = self.proj(x)  # B C Wl\n        if self.norm is not None:\n            # Wl = x.size(2)\n            x = x.transpose(1, 2)\n            x = self.norm(x)\n            # x = x.transpose(1, 2).view(-1, self.embed_dim, Wl)\n\n        return x\n\n\nclass SwinTransformer_1D(nn.Module):\n    r""" Swin Transformer\n        A PyTorch impl of : `Swin Transformer: Hierarchical Vision Transformer using Shifted Windows`  -\n          https://arxiv.org/pdf/2103.14030\n    Args:\n        img_size (int | tuple(int)): Input image size. Default 224\n        patch_size (int | tuple(int)): Patch size. Default: 4\n        in_chans (int): Number of input image channels. Default: 3\n        num_classes (int): Number of classes for classification head. Default: 1000\n        embed_dim (int): Patch embedding dimension. Default: 96\n        depths (tuple(int)): Depth of each Swin Transformer layer.\n        num_heads (tuple(int)): Number of attention heads in different layers.\n        window_size (int): Window size. Default: 7\n        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim. Default: 4\n        qkv_bias (bool): If True, add a learnable bias to query, key, value. Default: True\n        drop_rate (float): Dropout rate. Default: 0\n        attn_drop_rate (float): Attention dropout rate. Default: 0\n        drop_path_rate (float): Stochastic depth rate. Default: 0.1\n        norm_layer (nn.Module): Normalization layer. Default: nn.LayerNorm.\n        ape (bool): If True, add absolute position embedding to the patch embedding. Default: False\n        patch_norm (bool): If True, add normalization after patch embedding. Default: True\n        use_checkpoint (bool): Whether to use checkpointing to save memory. Default: False\n        pretrained_window_sizes (tuple(int)): Pretrained window sizes of each layer.\n    """\n\n    def __init__(self, seq_len=500, patch_size=4, in_chans=32, num_classes=4,\n                 embed_dim=96, depths=[2, 2, 6, 2], num_heads=[3, 6, 12, 24],\n                 window_size=7, mlp_ratio=4., qkv_bias=True,\n                 drop_rate=0., attn_drop_rate=0., drop_path_rate=0.1,\n                 norm_layer=nn.LayerNorm, ape=False, patch_norm=True,\n                 use_checkpoint=False, pretrained_window_sizes=[0, 0, 0, 0], **kwargs):\n        super().__init__()\n\n        # self.num_classes = num_classes\n        self.num_layers = len(depths)\n        self.embed_dim = embed_dim\n        self.ape = ape\n        self.patch_norm = patch_norm\n        self.num_features = int(embed_dim * 2 ** (self.num_layers - 1))\n        self.mlp_ratio = mlp_ratio\n\n        \n        # if num_classes is not None:\n        # self.head = nn.Linear(1024, num_classes)\n        # else:\n        #     self.head = None\n        \n        # split image into non-overlapping patches\n        self.patch_embed = PatchEmbed1D(seq_len=seq_len,\n            patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim,\n            norm_layer=norm_layer if self.patch_norm else None)\n        num_patches = self.patch_embed.num_patches\n        patches_resolution = self.patch_embed.patches_resolution\n        self.patches_resolution = patches_resolution\n\n        # absolute position embedding\n        if self.ape:\n            self.absolute_pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))\n            trunc_normal_(self.absolute_pos_embed, std=.02)\n\n        self.pos_drop = nn.Dropout(p=drop_rate)\n\n        # stochastic depth\n        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]  # stochastic depth decay rule\n\n        # build layers\n        self.layers = nn.ModuleList()\n        for i_layer in range(self.num_layers):\n            layer = BasicLayer(dim=int(embed_dim * 2 ** i_layer),\n                               input_resolution=math.ceil(patches_resolution / (2 ** i_layer)),\n                               depth=depths[i_layer],\n                               num_heads=num_heads[i_layer],\n                               window_size=window_size,\n                               mlp_ratio=self.mlp_ratio,\n                               qkv_bias=qkv_bias,\n                               drop=drop_rate, attn_drop=attn_drop_rate,\n                               drop_path=dpr[sum(depths[:i_layer]):sum(depths[:i_layer + 1])],\n                               norm_layer=norm_layer,\n                               downsample=PatchMerging if (i_layer < self.num_layers - 1) else None,\n                               use_checkpoint=use_checkpoint,\n                               pretrained_window_size=pretrained_window_sizes[i_layer])\n            self.layers.append(layer)\n\n        self.norm = norm_layer(self.num_features)\n        self.avgpool = nn.AdaptiveAvgPool1d(1)\n        self.head = nn.Linear(self.num_features, num_classes) if num_classes > 0 else nn.Identity()\n\n        self.apply(self._init_weights)\n        for bly in self.layers:\n            bly._init_respostnorm()\n\n    def _init_weights(self, m):\n        if isinstance(m, nn.Linear):\n            trunc_normal_(m.weight, std=.02)\n            if isinstance(m, nn.Linear) and m.bias is not None:\n                nn.init.constant_(m.bias, 0)\n        elif isinstance(m, nn.LayerNorm):\n            nn.init.constant_(m.bias, 0)\n            nn.init.constant_(m.weight, 1.0)\n\n    @torch.jit.ignore\n    def no_weight_decay(self):\n        return {\'absolute_pos_embed\'}\n\n    @torch.jit.ignore\n    def no_weight_decay_keywords(self):\n        return {"cpb_mlp", "logit_scale", \'relative_position_bias_table\'}\n\n    def forward_features(self, x):\n        x = self.patch_embed(x)\n        if self.ape:\n            x = x + self.absolute_pos_embed\n        x = self.pos_drop(x)\n\n        for layer in self.layers:\n            x = layer(x)\n\n        x = self.norm(x)  # B L C\n        x = self.avgpool(x.transpose(1, 2))  # B C 1\n        x = torch.flatten(x, 1)\n        return x\n\n    def forward(self, x):\n        x = self.forward_features(x)\n        x = self.head(x)\n        \n        # if return_class_scores and self.head is not None:\n        #     x = self.head(x)\n        return x\n\n\nif __name__ == "__main__":\n    net = SwinTransformerV2_1D(seq_len=2000, patch_size=4, in_chans=32, num_classes=9,\n                 embed_dim=96, depths=[2, 2, 6, 2], num_heads=[3, 6, 12, 24],\n                 window_size=31, mlp_ratio=4., qkv_bias=True,\n                 drop_rate=0., attn_drop_rate=0., drop_path_rate=0.1,\n                 norm_layer=nn.LayerNorm, ape=False, patch_norm=True,\n                 use_checkpoint=False, pretrained_window_sizes=[0, 0, 0, 0])\n    x = torch.rand(2, 32, 2000)\n    # x = torch.rand(2, 96, 64)\n    print(net(x).shape)\n', 'training/har_models.py': '"""Models shared by local training, Kaggle, evaluation and later export."""\nimport sys\nfrom pathlib import Path\n\nimport torch.nn as nn\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom models.SwinTransformer_cls import SwinTransformer_1D\n\n\nclass CNNBaseline(nn.Module):\n    def __init__(self, channels=(32, 64, 128), dropout=0.1):\n        super().__init__()\n        blocks = []\n        previous = 3\n        for width in channels:\n            blocks.extend([nn.Conv1d(previous, width, 5, padding=2), nn.BatchNorm1d(width),\n                           nn.ReLU(), nn.MaxPool1d(2)])\n            previous = width\n        self.features = nn.Sequential(*blocks)\n        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(previous, 6))\n\n    def forward(self, x):\n        return self.head(self.features(x))\n\n\ndef build_model(config):\n    if config[\'model\'] == \'swin\':\n        return SwinTransformer_1D(**config[\'model_kwargs\'])\n    if config[\'model\'] == \'cnn\':\n        return CNNBaseline(**config[\'model_kwargs\'])\n    raise ValueError(\'Unknown model: \' + config[\'model\'])\n', 'training/har_metrics.py': "import numpy as np\n\n\ndef classification_metrics(y_true, y_pred):\n    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)\n    if y_true.shape != y_pred.shape or y_true.ndim != 1 or not len(y_true):\n        raise ValueError('Expected matching non-empty one-dimensional labels')\n    if not np.isin(y_true, range(6)).all() or not np.isin(y_pred, range(6)).all():\n        raise ValueError('Labels must be 0..5')\n    matrix = np.zeros((6, 6), dtype=np.int64)\n    np.add.at(matrix, (y_true, y_pred), 1)\n    tp = matrix.diagonal().astype(float)\n    support = matrix.sum(axis=1)\n    precision = np.divide(tp, matrix.sum(axis=0), out=np.zeros(6), where=matrix.sum(axis=0) != 0)\n    recall = np.divide(tp, support, out=np.zeros(6), where=support != 0)\n    f1 = np.divide(2*precision*recall, precision+recall, out=np.zeros(6), where=precision+recall != 0)\n    return dict(accuracy=float(tp.sum()/len(y_true)), macro_f1=float(f1.mean()),\n                per_class=[dict(id=i, precision=float(precision[i]), recall=float(recall[i]),\n                                f1=float(f1[i]), support=int(support[i])) for i in range(6)],\n                confusion_matrix=matrix.tolist(), samples=len(y_true))\n", 'training/train_har.py': '"""Train using train/val ONLY. Evaluate official test in evaluate.py later."""\nimport argparse\nimport hashlib\nimport json\nimport math\nimport platform\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader, TensorDataset\n\nfrom har_metrics import classification_metrics\nfrom har_models import build_model\nfrom prepare_uci_har import ROOT, content_hash, write_json\n\n\ndef load_data(data_dir, split):\n    with np.load(data_dir / f\'{split}.npz\', allow_pickle=False) as archive:\n        X, y = archive[\'X\'].copy(), archive[\'y\'].copy()\n        subject, indices = archive[\'subject_id\'].copy(), archive[\'source_index\'].copy()\n    if X.dtype != np.float32 or X.shape[1:] != (3, 128) or y.dtype != np.int64:\n        raise ValueError(f\'{split}: tensor contract mismatch\')\n    if len(X) != len(y) or not np.isfinite(X).all() or not np.isin(y, range(6)).all():\n        raise ValueError(f\'{split}: invalid inputs or labels\')\n    return X, y, subject, indices\n\n\ndef validate_data(data_dir):\n    train, val = load_data(data_dir, \'train\'), load_data(data_dir, \'val\')\n    if set(train[2]) & set(val[2]):\n        raise ValueError(\'Train and validation subjects overlap\')\n    manifest = json.loads((data_dir / \'split-manifest.json\').read_text(encoding=\'utf-8\'))\n    for split, data in [(\'train\', train), (\'val\', val)]:\n        entry = manifest[\'splits\'][split]\n        if np.unique(data[2]).tolist() != entry[\'subject_ids\'] or data[3].tolist() != entry[\'source_indices_zero_based\']:\n            raise ValueError(\'Dataset differs from split manifest\')\n    return train, val\n\n\ndef seed_everything(seed, threads):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.set_num_threads(threads)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.use_deterministic_algorithms(True)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\n\n@torch.inference_mode()\ndef predict(model, loader, device):\n    model.eval()\n    predicted, truth, logits = [], [], []\n    total_loss = 0.0\n    for inputs, labels in loader:\n        output = model(inputs.to(device))\n        if not torch.isfinite(output).all():\n            raise ValueError(\'Non-finite logits\')\n        total_loss += torch.nn.functional.cross_entropy(output, labels.to(device), reduction=\'sum\').item()\n        predicted.extend(output.argmax(dim=1).cpu().tolist())\n        truth.extend(labels.tolist())\n        logits.append(output.cpu().numpy())\n    metrics = classification_metrics(truth, predicted)\n    metrics[\'loss\'] = total_loss / len(truth)\n    return metrics, np.concatenate(logits), np.array(predicted, dtype=np.int64)\n\n\ndef train(config, data_dir, output, device_name=\'auto\', threads=4, resume=False):\n    seed_everything(config[\'seed\'], threads)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\') if device_name == \'auto\' else torch.device(device_name)\n    train_data, val_data = validate_data(data_dir)\n    preprocessing = json.loads((data_dir / \'preprocessing.json\').read_text(encoding=\'utf-8\'))\n    labels = json.loads((data_dir / \'labels.json\').read_text(encoding=\'utf-8\'))\n    data_hash = content_hash(*train_data, *val_data)\n    output.mkdir(parents=True, exist_ok=True)\n    if (output / \'last.pth\').exists() and not resume:\n        raise ValueError(\'Run already exists. Use --resume or a new --output-dir.\')\n    model = build_model(config).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=config[\'lr\'], weight_decay=config[\'weight_decay\'])\n    def schedule(epoch):\n        if epoch < config[\'warmup_epochs\']:\n            return (epoch+1)/config[\'warmup_epochs\']\n        progress = (epoch-config[\'warmup_epochs\'])/max(1, config[\'epochs\']-config[\'warmup_epochs\'])\n        return 0.05 + 0.95*(1+math.cos(math.pi*progress))/2\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)\n    start_epoch, best_f1, best_epoch, stale = 0, -1.0, 0, 0\n    history = []\n    previous_seconds = 0\n    if resume:\n        # Only resume checkpoints produced locally/by this notebook, not untrusted uploads.\n        saved = torch.load(output / \'last.pth\', map_location=device, weights_only=False)\n        if saved[\'config\'] != config or saved[\'data_hash\'] != data_hash or saved[\'preprocessing\'] != preprocessing:\n            raise ValueError(\'Resume config/data/preprocessing differs from checkpoint\')\n        model.load_state_dict(saved[\'model_state_dict\'])\n        optimizer.load_state_dict(saved[\'optimizer_state_dict\'])\n        scheduler.load_state_dict(saved[\'scheduler_state_dict\'])\n        torch.set_rng_state(saved[\'torch_rng\'].cpu())\n        if device.type == \'cuda\' and saved[\'cuda_rng\']:\n            torch.cuda.set_rng_state_all([value.cpu() for value in saved[\'cuda_rng\']])\n        start_epoch, best_f1, best_epoch, stale = saved[\'epoch\'], saved[\'best_f1\'], saved[\'best_epoch\'], saved[\'stale\']\n        history = saved[\'history\']\n        previous_seconds = saved[\'training_seconds\']\n    val_loader = DataLoader(TensorDataset(torch.from_numpy(val_data[0]), torch.from_numpy(val_data[1])),\n                            batch_size=config[\'batch_size\'], shuffle=False)\n    train_dataset = TensorDataset(torch.from_numpy(train_data[0]), torch.from_numpy(train_data[1]))\n    metadata = dict(config=config, data_hash=data_hash, preprocessing=preprocessing, labels=labels,\n                    trained=True, parameters=sum(p.numel() for p in model.parameters()),\n                    environment=dict(device=str(device), hardware=torch.cuda.get_device_name() if device.type == \'cuda\' else platform.processor(),\n                                     python=platform.python_version(), torch=torch.__version__, numpy=np.__version__, threads=threads))\n    write_json(output / \'run-config.json\', metadata)\n    started = time.perf_counter()\n    for epoch in range(start_epoch, config[\'epochs\']):\n        if stale >= config[\'patience\']:\n            break\n        epoch_started = time.perf_counter()\n        # Epoch-specific shuffling survives interruption without hidden loader state.\n        loader = DataLoader(train_dataset, batch_size=config[\'batch_size\'], shuffle=True,\n                            generator=torch.Generator().manual_seed(config[\'seed\']+epoch))\n        model.train()\n        loss_sum = 0.0\n        for inputs, targets in loader:\n            inputs, targets = inputs.to(device), targets.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            loss = torch.nn.functional.cross_entropy(model(inputs), targets)\n            if not torch.isfinite(loss):\n                raise ValueError(\'Non-finite training loss\')\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), config[\'clip_grad_norm\'], error_if_nonfinite=True)\n            optimizer.step()\n            loss_sum += loss.item()*len(targets)\n        val_metrics, _, _ = predict(model, val_loader, device)\n        record = dict(epoch=epoch+1, train_loss=loss_sum/len(train_dataset), validation=val_metrics,\n                      lr=optimizer.param_groups[0][\'lr\'], epoch_seconds=time.perf_counter()-epoch_started)\n        history.append(record)\n        improved = val_metrics[\'macro_f1\'] > best_f1 + 1e-6\n        if improved:\n            best_f1, best_epoch, stale = val_metrics[\'macro_f1\'], epoch+1, 0\n            torch.save(dict(**metadata, epoch=epoch+1, validation=val_metrics, model_state_dict=model.state_dict()), output / \'best.pth\')\n        else:\n            stale += 1\n        scheduler.step()\n        elapsed = previous_seconds + time.perf_counter()-started\n        # Atomic checkpoint promotion preserves the previous last checkpoint if interrupted.\n        checkpoint_path = output / \'last.pth.partial\'\n        torch.save(dict(**metadata, epoch=epoch+1, best_f1=best_f1, best_epoch=best_epoch, stale=stale,\n                        history=history, training_seconds=elapsed, model_state_dict=model.state_dict(),\n                        optimizer_state_dict=optimizer.state_dict(), scheduler_state_dict=scheduler.state_dict(),\n                        torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all() if device.type == \'cuda\' else []), checkpoint_path)\n        checkpoint_path.replace(output / \'last.pth\')\n        write_json(output / \'history.json\', history)\n        print(f"{config[\'model\']} seed={config[\'seed\']} epoch={epoch+1} train_loss={record[\'train_loss\']:.4f} val_f1={best_f1:.4f} current_f1={val_metrics[\'macro_f1\']:.4f} seconds={record[\'epoch_seconds\']:.1f}", flush=True)\n    summary = dict(**metadata, best_epoch=best_epoch, best_validation_macro_f1=best_f1,\n                   epochs_completed=len(history), training_seconds=previous_seconds+time.perf_counter()-started,\n                   stopped_early=stale >= config[\'patience\'], test_used_for_training=False,\n                   checkpoint_sha256=hashlib.sha256((output / \'best.pth\').read_bytes()).hexdigest())\n    write_json(output / \'summary.json\', summary)\n    return summary\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--config\', type=Path, required=True)\n    parser.add_argument(\'--data-dir\', type=Path, default=ROOT / \'data/processed/uci-har-3ch-v1\')\n    parser.add_argument(\'--output-dir\', type=Path, required=True)\n    parser.add_argument(\'--device\', default=\'auto\', choices=[\'auto\', \'cpu\', \'cuda\'])\n    parser.add_argument(\'--threads\', type=int, default=4)\n    parser.add_argument(\'--resume\', action=\'store_true\')\n    args = parser.parse_args()\n    config = json.loads(args.config.read_text(encoding=\'utf-8\'))\n    train(config, args.data_dir, args.output_dir, args.device, args.threads, args.resume)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'training/evaluate_har.py': '"""Freeze validation-only selection BEFORE opening the official test split."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader, TensorDataset\n\nfrom har_models import build_model\nfrom prepare_uci_har import ROOT, content_hash, write_json\nfrom train_har import load_data, predict, seed_everything, validate_data\n\n\ndef sha256(path):\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef freeze_selection(runs, output, data_dir):\n    candidates = []\n    train_data, val_data = validate_data(data_dir)\n    expected_hash = content_hash(*train_data, *val_data)\n    preprocessing = json.loads((data_dir / \'preprocessing.json\').read_text(encoding=\'utf-8\'))\n    labels = json.loads((data_dir / \'labels.json\').read_text(encoding=\'utf-8\'))\n    for run in runs:\n        summary = json.loads((run / \'summary.json\').read_text(encoding=\'utf-8\'))\n        digest = sha256(run / \'best.pth\')\n        if digest != summary[\'checkpoint_sha256\']:\n            raise ValueError(\'Checkpoint differs from completed run: \' + str(run))\n        if summary[\'data_hash\'] != expected_hash or summary[\'preprocessing\'] != preprocessing or summary[\'labels\'] != labels:\n            raise ValueError(\'Run trained with a different dataset or contract\')\n        candidates.append(dict(run=run.name, model=summary[\'config\'][\'model\'], seed=summary[\'config\'][\'seed\'],\n                               validation_macro_f1=summary[\'best_validation_macro_f1\'], best_epoch=summary[\'best_epoch\'],\n                               checkpoint_sha256=digest))\n    if len({c[\'run\'] for c in candidates}) != len(candidates):\n        raise ValueError(\'Run names must be unique\')\n    selected = {}\n    for family in [\'cnn\', \'swin\']:\n        eligible = [c for c in candidates if c[\'model\'] == family]\n        if not eligible:\n            raise ValueError(\'Need both CNN baseline and Swin runs\')\n        selected[family] = max(eligible, key=lambda c: (c[\'validation_macro_f1\'], -c[\'seed\']))[\'run\']\n    selection = dict(rule=\'Best validation macro-F1 per family; tie uses smallest seed. Deployment family fixed to Swin.\',\n                     test_used_for_selection=False, candidates=candidates, selected=selected,\n                     deployment_run=selected[\'swin\'])\n    path = output / \'selection.json\'\n    if path.exists() and json.loads(path.read_text(encoding=\'utf-8\')) != selection:\n        raise ValueError(\'Selection already frozen. Do not change candidates after viewing test results.\')\n    write_json(path, selection)\n    return selection\n\n\ndef plot_run(run, metrics, output):\n    history = json.loads((run / \'history.json\').read_text(encoding=\'utf-8\'))\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)\n    epochs = [h[\'epoch\'] for h in history]\n    axes[0].plot(epochs, [h[\'train_loss\'] for h in history], label=\'Train\')\n    axes[0].plot(epochs, [h[\'validation\'][\'loss\'] for h in history], label=\'Validation\')\n    axes[0].set(xlabel=\'Epoch\', ylabel=\'Cross-entropy\', title=run.name)\n    axes[0].legend()\n    axes[1].plot(epochs, [h[\'validation\'][\'macro_f1\'] for h in history])\n    axes[1].set(xlabel=\'Epoch\', ylabel=\'Validation macro-F1\', ylim=(0, 1))\n    fig.savefig(output / f\'{run.name}-learning-curves.png\', dpi=140)\n    plt.close(fig)\n    matrix = np.asarray(metrics[\'confusion_matrix\'])\n    codes = [c[\'code\'] for c in metrics[\'per_class\']]\n    fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)\n    im = ax.imshow(matrix, cmap=\'Blues\')\n    ax.set(xticks=range(6), yticks=range(6), xticklabels=codes, yticklabels=codes,\n           xlabel=\'Predicted\', ylabel=\'True\', title=run.name + \' — official test (counts)\')\n    plt.setp(ax.get_xticklabels(), rotation=40, ha=\'right\')\n    for row in range(6):\n        for col in range(6):\n            ax.text(col, row, str(matrix[row, col]), ha=\'center\', va=\'center\',\n                    color=\'white\' if matrix[row, col] > matrix.max()/2 else \'black\')\n    fig.colorbar(im, ax=ax)\n    fig.savefig(output / f\'{run.name}-confusion-matrix.png\', dpi=140)\n    plt.close(fig)\n\n\ndef seed_statistics(results):\n    aggregates = {}\n    for family in [\'cnn\', \'swin\']:\n        members = [result for result in results if result[\'model\'] == family]\n        if not members:\n            continue\n        aggregates[family] = dict(seeds=[r[\'seed\'] for r in members], count=len(members),\n                                 std_ddof=1 if len(members) > 1 else None)\n        for metric in [\'accuracy\', \'macro_f1\']:\n            values = [result[\'test\'][metric] for result in members]\n            aggregates[family][metric] = dict(mean=float(np.mean(values)),\n                                              std=float(np.std(values, ddof=1)) if len(values) > 1 else None)\n    return aggregates\n\n\ndef evaluate(runs, data_dir, output, device=\'cpu\', all_seeds=False):\n    seed_everything(42, 4)\n    output.mkdir(parents=True, exist_ok=True)\n    selection = freeze_selection(runs, output, data_dir)\n    # Test is not opened until selection has been validated and written to disk.\n    X, y, subjects, indices = load_data(data_dir, \'test\')\n    train_data, val_data = validate_data(data_dir)\n    manifest = json.loads((data_dir / \'split-manifest.json\').read_text(encoding=\'utf-8\'))[\'splits\'][\'test\']\n    if set(subjects) & (set(train_data[2]) | set(val_data[2])):\n        raise ValueError(\'Test subjects overlap train/validation\')\n    if indices.tolist() != manifest[\'source_indices_zero_based\'] or np.unique(subjects).tolist() != manifest[\'subject_ids\']:\n        raise ValueError(\'Test differs from manifest\')\n    loader = DataLoader(TensorDataset(torch.from_numpy(X), torch.from_numpy(y)), batch_size=64, shuffle=False)\n    results = []\n    for run in runs:\n        if not all_seeds and run.name not in selection[\'selected\'].values():\n            continue\n        checkpoint = torch.load(run / \'best.pth\', map_location=device, weights_only=False)\n        model = build_model(checkpoint[\'config\']).to(device)\n        model.load_state_dict(checkpoint[\'model_state_dict\'], strict=True)\n        metrics, logits, predictions = predict(model, loader, torch.device(device))\n        for record, label in zip(metrics[\'per_class\'], checkpoint[\'labels\'][\'classes\']):\n            record.update(code=label[\'code\'], name_vi=label[\'name_vi\'])\n        result = dict(run=run.name, model=checkpoint[\'config\'][\'model\'], seed=checkpoint[\'config\'][\'seed\'],\n                      checkpoint_sha256=sha256(run / \'best.pth\'), best_epoch=checkpoint[\'epoch\'],\n                      validation=checkpoint[\'validation\'], test=metrics, test_sha256=content_hash(X, y, subjects, indices),\n                      environment=checkpoint[\'environment\'], trained=True, test_used_for_selection=False)\n        write_json(output / f\'{run.name}-metrics.json\', result)\n        np.savez_compressed(output / f\'{run.name}-predictions.npz\', y_true=y, y_pred=predictions,\n                            logits=logits, subject_id=subjects, source_index=indices)\n        plot_run(run, metrics, output)\n        results.append(result)\n        print(f"{run.name}: test accuracy={metrics[\'accuracy\']:.6f} macro_f1={metrics[\'macro_f1\']:.6f}", flush=True)\n    report = dict(selection=selection, results=results, target_macro_f1=0.90,\n                  target_met=next(r[\'test\'][\'macro_f1\'] for r in results if r[\'run\'] == selection[\'deployment_run\']) >= 0.90)\n    if all_seeds:\n        report[\'seed_statistics\'] = seed_statistics(results)\n        report[\'test_evaluation_scope\'] = \'All predeclared seeds; selection remains validation-only. Official test was already viewed in prior local run.\'\n    write_json(output / \'evaluation.json\', report)\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--runs\', nargs=\'+\', type=Path, required=True)\n    parser.add_argument(\'--data-dir\', type=Path, default=ROOT / \'data/processed/uci-har-3ch-v1\')\n    parser.add_argument(\'--output-dir\', type=Path, default=ROOT / \'artifacts/phase3/evaluation\')\n    parser.add_argument(\'--device\', choices=[\'cpu\', \'cuda\'], default=\'cpu\')\n    parser.add_argument(\'--all-seeds\', action=\'store_true\', help=\'Report all predeclared seeds, never select by test\')\n    args = parser.parse_args()\n    evaluate(args.runs, args.data_dir, args.output_dir, args.device, args.all_seeds)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'training/prepare_uci_har.py': '"""Prepare original UCI HAR windows. Requires only numpy, works on Kaggle/CPU.\n\nExample: python training/prepare_uci_har.py --download\nAlternatively --archive PATH or --raw-dir \'/kaggle/input/.../UCI HAR Dataset\'.\n"""\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nimport urllib.request\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\n\nROOT = Path(__file__).resolve().parents[1]\nLABEL_CODES = [\'WALKING\', \'WALKING_UPSTAIRS\', \'WALKING_DOWNSTAIRS\', \'SITTING\', \'STANDING\', \'LAYING\']\nLABEL_VI = [\'Đi bộ\', \'Lên cầu thang\', \'Xuống cầu thang\', \'Ngồi\', \'Đứng\', \'Nằm\']\n\n\ndef write_json(path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf-8\')\n\n\ndef content_hash(*arrays):\n    digest = hashlib.sha256()\n    for array in arrays:\n        array = np.ascontiguousarray(array)\n        digest.update(str(array.dtype).encode())\n        digest.update(str(array.shape).encode())\n        digest.update(array.tobytes())\n    return digest.hexdigest()\n\n\nclass Source:\n    """Read only required members; never extract arbitrary ZIP paths."""\n    def __init__(self, path):\n        self.path = Path(path)\n        self.archive = None\n        self.nested_buffer = None\n        if self.path.is_dir():\n            if not (self.path / \'train\').is_dir():\n                self.path = self.path / \'UCI HAR Dataset\'\n            if not (self.path / \'train\').is_dir():\n                raise ValueError(\'raw-dir must contain the original UCI HAR Dataset/train directory\')\n        else:\n            self.archive = zipfile.ZipFile(self.path)\n            if not any(name.endswith(\'/train/y_train.txt\') for name in self.archive.namelist()):\n                nested = [name for name in self.archive.namelist() if name.endswith(\'UCI HAR Dataset.zip\')]\n                if len(nested) != 1:\n                    raise ValueError(\'Archive does not contain the original UCI HAR windows\')\n                self.nested_buffer = io.BytesIO(self.archive.read(nested[0]))\n                self.archive.close()\n                self.archive = zipfile.ZipFile(self.nested_buffer)\n\n    def read(self, relative):\n        if self.archive is None:\n            return (self.path / relative).read_bytes()\n        matches = [name for name in self.archive.namelist()\n                   if name == relative or name.endswith(\'/\' + relative)]\n        if len(matches) != 1:\n            raise ValueError(f\'Expected exactly one archive member: {relative}\')\n        return self.archive.read(matches[0])\n\n    def close(self):\n        if self.archive:\n            self.archive.close()\n        if self.nested_buffer:\n            self.nested_buffer.close()\n\n\ndef load_split(source, split, config):\n    signal_files = [f\'{split}/Inertial Signals/{channel}_{split}.txt\' for channel in config[\'channels\']]\n    blobs = [source.read(name) for name in signal_files]\n    label_blob = source.read(f\'{split}/y_{split}.txt\')\n    subject_blob = source.read(f\'{split}/subject_{split}.txt\')\n    X = np.stack([np.loadtxt(io.BytesIO(blob), dtype=np.float32, ndmin=2) for blob in blobs], axis=1)\n    # Read IDs as floats first to reject non-integral IDs instead of truncating.\n    labels = np.loadtxt(io.BytesIO(label_blob), ndmin=1)\n    subjects = np.loadtxt(io.BytesIO(subject_blob), ndmin=1)\n    if X.shape != (len(labels), 3, 128) or subjects.shape != labels.shape:\n        raise ValueError(f\'{split}: mismatched signal, label or subject shape\')\n    if not np.isfinite(X).all() or not np.isfinite(labels).all() or not np.isfinite(subjects).all():\n        raise ValueError(f\'{split}: non-finite values\')\n    if not np.isin(labels, np.arange(1, 7)).all():\n        raise ValueError(f\'{split}: activity IDs must be integers 1..6\')\n    if not np.isin(subjects, np.arange(1, 31)).all():\n        raise ValueError(f\'{split}: subject IDs must be integers 1..30\')\n    hashes = dict(zip(signal_files, [hashlib.sha256(blob).hexdigest() for blob in blobs]))\n    hashes[f\'{split}/y_{split}.txt\'] = hashlib.sha256(label_blob).hexdigest()\n    hashes[f\'{split}/subject_{split}.txt\'] = hashlib.sha256(subject_blob).hexdigest()\n    return X, labels.astype(np.int64) - 1, subjects.astype(np.int64), hashes\n\n\ndef split_subjects(subjects, seed, validation_subject_count):\n    unique = np.unique(subjects)\n    if not 0 < validation_subject_count < len(unique):\n        raise ValueError(\'validation_subject_count must leave both train and validation subjects\')\n    selected = np.sort(np.random.default_rng(seed).permutation(unique)[:validation_subject_count])\n    mask = np.isin(subjects, selected)\n    return np.flatnonzero(~mask), np.flatnonzero(mask), selected\n\n\ndef fit_preprocessing(train, config):\n    if train.ndim != 3 or train.shape[1:] != (3, 128) or not len(train) or not np.isfinite(train).all():\n        raise ValueError(\'Expected finite non-empty training windows [N,3,128]\')\n    # Accumulate in float64, store float32 values: same numbers used by web.\n    mean = train.mean(axis=(0, 2), dtype=np.float64).astype(np.float32)\n    std = train.std(axis=(0, 2), dtype=np.float64, ddof=0).astype(np.float32)\n    std = np.maximum(std, np.float32(config[\'epsilon\']))\n    return mean, std\n\n\ndef normalize(X, mean, std):\n    return ((X - mean[None, :, None]) / std[None, :, None]).astype(np.float32)\n\n\ndef prepare(source_path, output, metadata_dir, demo_dir, config):\n    if config[\'channels\'] != [\'total_acc_x\', \'total_acc_y\', \'total_acc_z\'] or config[\'window_size\'] != 128:\n        raise ValueError(\'Phase 2 contract requires total_acc X/Y/Z and 128-point original windows\')\n    if config[\'input_unit\'] != \'g\' or config[\'includes_gravity\'] is not True or config[\'sample_rate_hz\'] != 50:\n        raise ValueError(\'Original UCI HAR uses total acceleration in g at 50 Hz\')\n    if not np.isfinite(config[\'epsilon\']) or config[\'epsilon\'] <= 0:\n        raise ValueError(\'epsilon must be positive and finite\')\n    source = Source(source_path)\n    try:\n        activity_blob = source.read(\'activity_labels.txt\')\n        activities = [line.split() for line in activity_blob.decode(\'ascii\').splitlines() if line.strip()]\n        if activities != [[str(i+1), code] for i, code in enumerate(LABEL_CODES)]:\n            raise ValueError(\'Source activity mapping differs from the Phase 0 contract\')\n        original_X, original_y, original_subjects, train_hashes = load_split(source, \'train\', config)\n        test_X, test_y, test_subjects, test_hashes = load_split(source, \'test\', config)\n    finally:\n        source.close()\n    if len(original_X) != 7352 or len(test_X) != 2947:\n        raise ValueError(\'Expected original UCI HAR split: 7352 train and 2947 test windows\')\n    if len(np.unique(original_subjects)) != 21 or len(np.unique(test_subjects)) != 9:\n        raise ValueError(\'Expected original subject split: 21 train and 9 test subjects\')\n    if set(original_subjects) & set(test_subjects):\n        raise ValueError(\'Official train/test subjects overlap\')\n\n    train_indices, val_indices, val_subjects = split_subjects(original_subjects, config[\'seed\'], config[\'validation_subject_count\'])\n    mean, std = fit_preprocessing(original_X[train_indices], config)\n    specs = {\'train\': (original_X[train_indices], original_y[train_indices], original_subjects[train_indices], train_indices, \'train\'),\n             \'val\': (original_X[val_indices], original_y[val_indices], original_subjects[val_indices], val_indices, \'train\'),\n             \'test\': (test_X, test_y, test_subjects, np.arange(len(test_y)), \'test\')}\n    output.mkdir(parents=True, exist_ok=True)\n    manifest_splits, report_splits = {}, {}\n    for name, (raw, y, subject_ids, source_indices, original_split) in specs.items():\n        if set(np.unique(y)) != set(range(6)):\n            raise ValueError(f\'{name}: every activity must be represented\')\n        X = normalize(raw, mean, std)\n        np.savez(output / f\'{name}.npz\', X=X, X_raw=raw, y=y, subject_id=subject_ids,\n                 source_index=source_indices.astype(np.int64))\n        manifest_splits[name] = dict(original_split=original_split, subject_ids=np.unique(subject_ids).tolist(),\n                                    source_indices_zero_based=source_indices.tolist(), count=len(y))\n        report_splits[name] = dict(shape=list(X.shape), dtype=str(X.dtype), label_dtype=str(y.dtype),\n                                  windows=len(y), subjects=np.unique(subject_ids).tolist(), finite=bool(np.isfinite(X).all()),\n                                  class_counts={LABEL_CODES[i]: int((y == i).sum()) for i in range(6)},\n                                  normalized_mean=X.mean(axis=(0, 2), dtype=np.float64).tolist(),\n                                  normalized_std=X.std(axis=(0, 2), dtype=np.float64).tolist(),\n                                  raw_min=raw.min(axis=(0, 2)).tolist(), raw_max=raw.max(axis=(0, 2)).tolist(),\n                                  content_sha256=content_hash(X, raw, y, subject_ids, source_indices.astype(np.int64)))\n\n    preprocessing = {key: config[key] for key in [\'contract_version\', \'artifact_version\', \'channels\', \'input_unit\', \'includes_gravity\', \'sample_rate_hz\', \'window_size\', \'stride\', \'epsilon\']}\n    preprocessing.update(mean=mean.tolist(), std=std.tolist(), input_layout=\'NCT\', fitted_on=\'train only after subject split\',\n                         fit_subject_ids=np.unique(original_subjects[train_indices]).tolist(), std_ddof=0,\n                         statistics_weighting=\'All time points across training windows, including original overlapping windows\',\n                         transformed_dtype=\'float32\')\n    labels = dict(contract_version=config[\'contract_version\'], artifact_version=config[\'artifact_version\'],\n                  classes=[dict(id=i, uci_id=i+1, code=code, name_vi=LABEL_VI[i]) for i, code in enumerate(LABEL_CODES)])\n    manifest = dict(artifact_version=config[\'artifact_version\'], config=config, splits=manifest_splits,\n                    source_file_sha256={**train_hashes, **test_hashes,\n                        \'activity_labels.txt\': hashlib.sha256(activity_blob).hexdigest()}, source_order_preserved=True)\n    write_json(metadata_dir / \'preprocessing.json\', preprocessing)\n    write_json(metadata_dir / \'labels.json\', labels)\n    write_json(metadata_dir / \'split-manifest.json\', manifest)\n    write_json(output / \'preprocessing.json\', preprocessing)\n    write_json(output / \'labels.json\', labels)\n    write_json(output / \'split-manifest.json\', manifest)\n\n    # First official TRAIN row for each label, chosen without any model prediction.\n    # Keep demo subjects out of test to avoid showcasing the held-out evaluation.\n    demo_dir.mkdir(parents=True, exist_ok=True)\n    demos = []\n    for label_id, code in enumerate(LABEL_CODES):\n        index = int(np.flatnonzero(original_y == label_id)[0])\n        filename = code.lower() + \'.csv\'\n        with (demo_dir / filename).open(\'w\', newline=\'\', encoding=\'utf-8\') as file:\n            writer = csv.writer(file)\n            writer.writerow([\'timestamp_ms\', \'acc_x\', \'acc_y\', \'acc_z\'])\n            for timestep in range(128):\n                writer.writerow([timestep*20, *[format(float(value), \'.9g\') for value in original_X[index, :, timestep]]])\n        demos.append(dict(file=filename, dataset=\'UCI HAR\', original_split=\'train\',\n                          prepared_split=\'val\' if index in val_indices else \'train\',\n                          subject_id=int(original_subjects[index]), source_index_zero_based=index,\n                          source_line_one_based=index+1, reference_label_id=label_id, reference_label=code,\n                          input_unit=\'g\', includes_gravity=True, window_size=128,\n                          timestamp_note=\'Relative display timestamps for one independent window; not a reconstructed continuous trial\',\n                          csv_sha256=hashlib.sha256((demo_dir / filename).read_bytes()).hexdigest()))\n    write_json(demo_dir / \'index.json\', dict(artifact_version=config[\'artifact_version\'],\n               dataset_url=\'https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones\',\n               selection_rule=\'First official train row per class, no model/predictions used\', samples=demos))\n    report = dict(status=\'passed\', artifact_version=config[\'artifact_version\'], source_url=config[\'source_url\'],\n                  source_archive_sha256=hashlib.sha256(Path(source_path).read_bytes()).hexdigest() if Path(source_path).is_file() else None,\n                  seed=config[\'seed\'], validation_subjects=val_subjects.tolist(),\n                  checks=dict(expected_official_counts=True, subject_disjoint=True, all_classes_present=True,\n                              finite_inputs=True, original_order_preserved=True, fit_train_only=True),\n                  splits=report_splits, preprocessing=preprocessing, demo_selection=\'First official train row per class\')\n    write_json(metadata_dir / \'data-report.json\', report)\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    source_group = parser.add_mutually_exclusive_group()\n    source_group.add_argument(\'--download\', action=\'store_true\')\n    source_group.add_argument(\'--archive\', type=Path)\n    source_group.add_argument(\'--raw-dir\', type=Path)\n    parser.add_argument(\'--config\', type=Path, default=ROOT / \'training/configs/data.json\')\n    parser.add_argument(\'--output-dir\', type=Path, default=ROOT / \'data/processed/uci-har-3ch-v1\')\n    parser.add_argument(\'--metadata-dir\', type=Path, default=ROOT / \'artifacts/phase2\')\n    parser.add_argument(\'--demo-dir\', type=Path, default=ROOT / \'web/public/samples/uci-har-3ch-v1\')\n    args = parser.parse_args()\n    config = json.loads(args.config.read_text(encoding=\'utf-8\'))\n    cache = ROOT / \'data/raw/uci-har.zip\'\n    source_path = args.raw_dir or args.archive or cache\n    if args.download and not cache.exists():\n        cache.parent.mkdir(parents=True, exist_ok=True)\n        partial = cache.with_suffix(\'.zip.partial\')\n        print(\'Downloading official UCI archive...\', flush=True)\n        request = urllib.request.Request(config[\'source_url\'], headers={\'User-Agent\': \'DuST-HAR/0.1\'})\n        with urllib.request.urlopen(request, timeout=120) as response, partial.open(\'wb\') as file:\n            while chunk := response.read(1024 * 1024):\n                file.write(chunk)\n        if not zipfile.is_zipfile(partial):\n            raise ValueError(\'Download is not a ZIP; cached file was not replaced\')\n        partial.replace(cache)\n    if not source_path.exists():\n        parser.error(\'Provide --download, --archive ZIP, or --raw-dir original dataset folder\')\n    report = prepare(source_path, args.output_dir, args.metadata_dir, args.demo_dir, config)\n    print(json.dumps({name: {\'shape\': value[\'shape\'], \'subjects\': value[\'subjects\']} for name, value in report[\'splits\'].items()}, indent=2))\n    print(\'PASS: saved prepared windows, metadata, manifest and 6 demo CSVs.\')\n\n\nif __name__ == \'__main__\':\n    main()\n', 'training/verify_phase2.py': '"""Verify actual prepared UCI arrays, provenance, demos and reproducibility."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport tempfile\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom prepare_uci_har import ROOT, Source, load_split, prepare, write_json\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--source\', type=Path, default=ROOT / \'data/raw/uci-har.zip\')\n    parser.add_argument(\'--prepared\', type=Path, default=ROOT / \'data/processed/uci-har-3ch-v1\')\n    parser.add_argument(\'--metadata\', type=Path, default=ROOT / \'artifacts/phase2\')\n    parser.add_argument(\'--demos\', type=Path, default=ROOT / \'web/public/samples/uci-har-3ch-v1\')\n    parser.add_argument(\'--config\', type=Path, default=ROOT / \'training/configs/data.json\')\n    args = parser.parse_args()\n    config = json.loads(args.config.read_text(encoding=\'utf-8\'))\n    manifest = json.loads((args.metadata / \'split-manifest.json\').read_text(encoding=\'utf-8\'))\n    preprocessing = json.loads((args.metadata / \'preprocessing.json\').read_text(encoding=\'utf-8\'))\n    report = json.loads((args.metadata / \'data-report.json\').read_text(encoding=\'utf-8\'))\n    source = Source(args.source)\n    try:\n        original = {name: load_split(source, name, config) for name in [\'train\', \'test\']}\n    finally:\n        source.close()\n    arrays = {}\n    for split in [\'train\', \'val\', \'test\']:\n        with np.load(args.prepared / f\'{split}.npz\', allow_pickle=False) as file:\n            arrays[split] = {name: file[name] for name in file.files}\n        data = arrays[split]\n        info = manifest[\'splits\'][split]\n        indices = np.asarray(info[\'source_indices_zero_based\'])\n        raw, labels, subjects, _ = original[info[\'original_split\']]\n        np.testing.assert_array_equal(data[\'X_raw\'], raw[indices])\n        np.testing.assert_array_equal(data[\'y\'], labels[indices])\n        np.testing.assert_array_equal(data[\'subject_id\'], subjects[indices])\n        np.testing.assert_array_equal(data[\'source_index\'], indices)\n        assert data[\'X\'].dtype == np.float32 and data[\'y\'].dtype == np.int64\n        assert data[\'X\'].shape[1:] == (3, 128) and np.isfinite(data[\'X\']).all()\n        assert set(np.unique(data[\'y\'])) == set(range(6))\n        # Independent transformation with metadata as a future browser would do.\n        expected = ((raw[indices].astype(np.float64) - np.array(preprocessing[\'mean\'])[None, :, None])\n                    / np.array(preprocessing[\'std\'])[None, :, None]).astype(np.float32)\n        np.testing.assert_allclose(data[\'X\'], expected, rtol=1e-6, atol=1e-6)\n    for a, b in [(\'train\', \'val\'), (\'train\', \'test\'), (\'val\', \'test\')]:\n        assert not set(arrays[a][\'subject_id\']) & set(arrays[b][\'subject_id\'])\n    combined = np.sort(np.concatenate([arrays[\'train\'][\'source_index\'], arrays[\'val\'][\'source_index\']]))\n    np.testing.assert_array_equal(combined, np.arange(7352))\n    np.testing.assert_array_equal(arrays[\'test\'][\'source_index\'], np.arange(2947))\n    train = arrays[\'train\'][\'X_raw\'].astype(np.float64)\n    np.testing.assert_array_equal(np.asarray(preprocessing[\'mean\'], dtype=np.float32), train.mean(axis=(0, 2)).astype(np.float32))\n    np.testing.assert_array_equal(np.asarray(preprocessing[\'std\'], dtype=np.float32), train.std(axis=(0, 2)).astype(np.float32))\n    np.testing.assert_allclose(arrays[\'train\'][\'X\'].mean(axis=(0, 2), dtype=np.float64), 0, atol=1e-6)\n    np.testing.assert_allclose(arrays[\'train\'][\'X\'].std(axis=(0, 2), dtype=np.float64), 1, atol=1e-6)\n\n    demo_index = json.loads((args.demos / \'index.json\').read_text(encoding=\'utf-8\'))\n    assert len(demo_index[\'samples\']) == 6\n    for demo in demo_index[\'samples\']:\n        path = args.demos / demo[\'file\']\n        with path.open(encoding=\'utf-8\', newline=\'\') as file:\n            rows = list(csv.DictReader(file))\n        values = np.asarray([[float(row[name]) for row in rows] for name in [\'acc_x\', \'acc_y\', \'acc_z\']], dtype=np.float32)\n        index = demo[\'source_index_zero_based\']\n        np.testing.assert_array_equal(values, original[\'train\'][0][index])\n        np.testing.assert_array_equal([int(row[\'timestamp_ms\']) for row in rows], np.arange(128)*20)\n        assert demo[\'reference_label_id\'] == int(original[\'train\'][1][index])\n        assert demo[\'subject_id\'] == int(original[\'train\'][2][index])\n        assert index == int(np.flatnonzero(original[\'train\'][1] == demo[\'reference_label_id\'])[0])\n        assert hashlib.sha256(path.read_bytes()).hexdigest() == demo[\'csv_sha256\']\n\n    # Rebuild all artifacts in a disposable task directory; compare numerical\n    # payloads because ZIP container timestamps need not be byte-identical.\n    with tempfile.TemporaryDirectory(prefix=\'dust-phase2-verify-\') as directory:\n        target = Path(directory)\n        repeat_report = prepare(args.source, target / \'data\', target / \'metadata\', target / \'demos\', config)\n        assert report == repeat_report\n        for filename in [\'split-manifest.json\', \'preprocessing.json\', \'labels.json\']:\n            assert (args.metadata / filename).read_bytes() == (target / \'metadata\' / filename).read_bytes()\n        for split, data in arrays.items():\n            with np.load(target / \'data\' / f\'{split}.npz\', allow_pickle=False) as repeat:\n                for name, expected in data.items():\n                    np.testing.assert_array_equal(expected, repeat[name])\n        for file in args.demos.iterdir():\n            if file.suffix in [\'.csv\', \'.json\']:\n                assert file.read_bytes() == (target / \'demos\' / file.name).read_bytes()\n    verification = dict(status=\'passed\', criteria=[\'P2-01\', \'P2-02\', \'P2-03\', \'P2-04\'],\n                        checks=[\'source pairing\', \'train/val/test subject disjoint\', \'complete source indices\',\n                                \'train-only mean/std\', \'normalized tensor matches metadata\',\n                                \'six CSVs roundtrip exactly to source float32\', \'demo selection independent of model\',\n                                \'second build identical arrays/metadata/demos\'],\n                        archive_sha256=report[\'source_archive_sha256\'], numpy_version=np.__version__)\n    write_json(args.metadata / \'verification.json\', verification)\n    print(json.dumps(verification, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'training/configs/data.json': '{\n  "dataset": "UCI HAR",\n  "dataset_version": "uci-har-original",\n  "contract_version": "1",\n  "artifact_version": "uci-har-3ch-v1",\n  "source_url": "https://archive.ics.uci.edu/static/public/240/human%2Bactivity%2Brecognition%2Busing%2Bsmartphones.zip",\n  "seed": 42,\n  "validation_subject_count": 5,\n  "channels": ["total_acc_x", "total_acc_y", "total_acc_z"],\n  "input_unit": "g",\n  "includes_gravity": true,\n  "sample_rate_hz": 50,\n  "window_size": 128,\n  "stride": 64,\n  "epsilon": 1e-8\n}\n', 'training/configs/train-cnn.json': '{\n  "experiment": "cnn-baseline-v1", "model": "cnn", "seed": 42,\n  "epochs": 60, "patience": 12, "batch_size": 64, "lr": 0.001,\n  "weight_decay": 0.01, "warmup_epochs": 3, "clip_grad_norm": 1.0,\n  "model_kwargs": {"channels": [32, 64, 128], "dropout": 0.1}\n}\n', 'training/configs/train-swin.json': '{\n  "experiment": "swin-small-v1", "model": "swin", "seed": 42,\n  "epochs": 60, "patience": 12, "batch_size": 64, "lr": 0.0003,\n  "weight_decay": 0.02, "warmup_epochs": 3, "clip_grad_norm": 1.0,\n  "model_kwargs": {\n    "seq_len": 128, "in_chans": 3, "num_classes": 6,\n    "patch_size": 2, "window_size": 8, "embed_dim": 32,\n    "depths": [2, 2, 2, 2], "num_heads": [2, 4, 8, 16],\n    "drop_rate": 0.1, "drop_path_rate": 0.1\n  }\n}\n'}
import hashlib
source_hashes = {}
for relative, source in SOURCES.items():
    path = WORK_ROOT / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.read_text(encoding='utf-8') != source:
        raise ValueError(f'Source differs: {relative}. Use a new WORK_ROOT.')
    path.write_text(source, encoding='utf-8', newline='\n')
    source_hashes[relative] = hashlib.sha256(source.encode('utf-8')).hexdigest()
(WORK_ROOT / 'source-hashes.json').write_text(json.dumps(source_hashes, indent=2), encoding='utf-8')
print('Embedded project files:', len(SOURCES))


## 2. Môi trường
Kaggle thường có sẵn PyTorch. Không cài lại CUDA/PyTorch để tránh tải nặng.
Ghi version thực tế; bản CPU tham chiếu của repo: Python 3.11, torch 2.5.1, numpy 1.26.4.
Môi trường khác có thể cho kết quả khác dù seed giống nhau. Matplotlib chỉ dùng vẽ báo cáo.


In [ ]:
import importlib.util
if importlib.util.find_spec('matplotlib') is None:
    run('-m', 'pip', 'install', 'matplotlib==3.9.4')
import torch, numpy as np
assert int(torch.__version__.split('.')[0]) >= 2, 'Need PyTorch 2+'
print({'python': sys.version, 'torch': torch.__version__, 'numpy': np.__version__,
       'cuda': torch.cuda.is_available(),
       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})


## 3. Chuẩn bị và kiểm chứng dữ liệu
Chạy lại Phase 2; kiểm tra source pairing, normalization và tính tái lập.


In [ ]:
data_dir = WORK_ROOT / 'data/processed/uci-har-3ch-v1'
source_args = ['--download'] if DATA_SOURCE is None else [
    '--raw-dir' if Path(DATA_SOURCE).is_dir() else '--archive', DATA_SOURCE]
run('training/prepare_uci_har.py', *source_args)
source_path = Path(DATA_SOURCE) if DATA_SOURCE else WORK_ROOT / 'data/raw/uci-har.zip'
run('training/verify_phase2.py', '--source', source_path)


## 4. Train CNN và Swin 1D
60 epochs tối đa, patience 12; AdamW, warmup/cosine, gradient clipping.
Không augmentation/sampling theo test. `X` trong NPZ đã normalize, không normalize lần hai.
Lưu best.pth, last.pth, config, history, summary từng run. Chạy lại cùng thư mục dùng resume.
Muốn resume giữa hai phiên Kaggle: tải và khôi phục cả thư mục project (đặc biệt last.pth),
hoặc giữ output bằng Save Version. Notebook không tự tìm checkpoint ở /kaggle/input.


In [ ]:
runs = []
for family in ['cnn', 'swin']:
    base = json.loads((WORK_ROOT / f'training/configs/train-{family}.json').read_text(encoding='utf-8'))
    for seed in SEEDS:
        config = dict(base, seed=int(seed))
        config_path = WORK_ROOT / f'training/configs/{family}-seed{seed}.json'
        if config_path.exists() and json.loads(config_path.read_text(encoding='utf-8')) != config:
            raise ValueError('Run config changed: use a new run directory')
        config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')
        output = WORK_ROOT / f'artifacts/phase3/runs/{family}-seed{seed}'
        args = ['training/train_har.py', '--config', config_path, '--output-dir', output, '--device', DEVICE]
        if (output / 'last.pth').exists():
            args.append('--resume')
        run(*args)
        runs.append(output)


## 5. Đóng băng lựa chọn, đánh giá test một lần
Chọn seed tốt nhất theo validation trong mỗi family, giữ Swin làm model triển khai.
selection.json không cho thay candidates sau khi đã xem test. Nếu dưới 0.90 macro-F1,
đọc confusion matrix, ghi giới hạn demo nghiên cứu; không tune lại theo test để tuyên bố benchmark.


In [ ]:
evaluation = WORK_ROOT / 'artifacts/phase3/evaluation'
run('training/evaluate_har.py', '--runs', *runs, '--output-dir', evaluation)
report = json.loads((evaluation / 'evaluation.json').read_text(encoding='utf-8'))
for result in report['results']:
    print(result['run'], 'accuracy', result['test']['accuracy'], 'macro-F1', result['test']['macro_f1'])
    print('Recall per class:', [(c['code'], c['recall']) for c in result['test']['per_class']])
print('Swin target >=0.90:', report['target_met'])
from IPython.display import display, Image
for path in sorted(evaluation.glob('*.png')):
    display(Image(filename=str(path)))


## 6. Tải artifact cho Phase 4
ZIP gồm best checkpoints, config/metadata, logs, predictions, biểu đồ và source hash.
Không gồm dữ liệu gốc hay last.pth/optimizer để nhẹ hơn; **không phải ZIP resume**.
best.pth chứa trained=true, labels và normalization thật. Chưa phải ONNX production.
Save Version để giữ output; tải ZIP về repo và kiểm tra hash trước bước export.


In [ ]:
archive = WORK_ROOT.parent / 'phase3-results.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted((WORK_ROOT / 'artifacts').rglob('*')):
        if path.is_file() and path.name not in ['last.pth', 'last.pth.partial']:
            z.write(path, path.relative_to(WORK_ROOT))
    z.write(WORK_ROOT / 'source-hashes.json', 'source-hashes.json')
from IPython.display import FileLink
display(FileLink(str(archive)))
print('Checkpoint for deployment:', report['selection']['deployment_run'])
